# Proyecto Integrador Corte 2 — PENGWIN · Semana 1 (semana 8 del curso)

**Sistema de detección, segmentación y clasificación de fragmentos pélvicos en CT**
Analítica de Datos · UAO 2026-2 · Prof. Carlos A. Ferro

> ⚠️ **Uso exclusivamente académico.** Esto no es un dispositivo médico, no está
> validado clínicamente y no sirve para apoyar decisiones quirúrgicas reales.

Esta primera semana no entrené nada. Me dediqué a dejar los datos en un estado
en el que pueda confiar, porque si el dataset viene torcido el modelo de las
semanas siguientes aprende lo torcido y uno no se da cuenta hasta el final.

Lo que entrego aquí:

| # | Entregable | Sección |
|---|---|---|
| 1 | Pipeline de carga del volumen + espaciado físico del header | §1 |
| 2 | Dataset curado: control de calidad CT ↔ máscara | §2 |
| 3 | Análisis exploratorio (EDA) de fragmentos por caso | §3 |
| 4 | Splits fijos train/val/test por paciente | §4 |
| 5 | Ventaneo HU funcionando | §5 |
| 6 | Visualizador 1: MIP del hueso por umbral HU | §6 |

Trabajé con los 100 casos de entrenamiento del challenge. Los 100 pasaron el
control de calidad y de ahí salieron 575 fragmentos para analizar.

## 0. Entorno

Esto corre igual en **Colab** y en local. En Colab monta el Drive, porque los
volúmenes pesan y no tiene sentido subirlos cada vez.

Si trabaja en Colab, la carpeta `PENGWIN-Corte2/` (con `pengwin/` adentro) tiene
que estar en el Drive del equipo, al lado de `PENGWIN_CT_train_images part1` y
`Part2`. Un detalle que me costó tiempo: si la carpeta está *compartida contigo*
y no en Mi unidad, Colab no la ve. Hay que hacer clic derecho →
*Organizar → Agregar acceso directo* primero.

In [1]:
import os, sys, glob, json, zipfile, urllib.request
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip -q install SimpleITK plotly

def buscar(patron, raices):
    for r in raices:
        hits = sorted(glob.glob(os.path.join(r, patron), recursive=True))
        if hits:
            return Path(hits[0])
    return None

if EN_COLAB:
    RAICES = ["/content/drive/MyDrive", "/content/drive/Shareddrives"]
    DIR_IMG_1 = buscar("**/PENGWIN_CT_train_images part1", RAICES)
    DIR_IMG_2 = buscar("**/PENGWIN_CT_train_images Part2", RAICES)
    RAIZ = buscar("**/PENGWIN-Corte2", RAICES) or Path("/content/PENGWIN-Corte2")
    DIRS_IMAGENES = [d for d in (DIR_IMG_1, DIR_IMG_2) if d]
    DIR_ETIQUETAS = Path("/content/labels")          # disco local de Colab (13 GB descomprimido)
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DIRS_IMAGENES = [RAIZ / "data/raw/images"]
    DIR_ETIQUETAS = RAIZ / "data/raw/labels"

sys.path.insert(0, str(RAIZ))
SALIDAS = RAIZ / "salidas"; SALIDAS.mkdir(exist_ok=True)
print("Proyecto :", RAIZ)
print("Imágenes :", DIRS_IMAGENES)
print("Máscaras :", DIR_ETIQUETAS)

Proyecto : /home/caesar/Projects/PENGWIN-Corte2
Imágenes : [PosixPath('/home/caesar/Projects/PENGWIN-Corte2/data/raw/images')]
Máscaras : /home/caesar/Projects/PENGWIN-Corte2/data/raw/labels


In [2]:
# Las máscaras NO están en el Drive: se bajan de Zenodo (registro 10927452, 33 MB comprimidas).
URL_LABELS = "https://zenodo.org/api/records/10927452/files/PENGWIN_CT_train_labels.zip/content"
if not any(DIR_ETIQUETAS.glob("*.mha")):
    zip_local = DIR_ETIQUETAS.parent / "PENGWIN_CT_train_labels.zip"
    if not zip_local.exists():
        urllib.request.urlretrieve(URL_LABELS, zip_local)
    zipfile.ZipFile(zip_local).extractall(DIR_ETIQUETAS)

RUTA_CT = {p.stem: p for d in DIRS_IMAGENES for p in d.glob("*.mha")}
RUTA_LB = {p.stem: p for p in DIR_ETIQUETAS.glob("*.mha")}
print(f"CT disponibles: {len(RUTA_CT)} · máscaras: {len(RUTA_LB)} · con ambas: {len(RUTA_CT.keys() & RUTA_LB.keys())}")

CT disponibles: 3 · máscaras: 100 · con ambas: 3


In [3]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import SimpleITK as sitk
from pengwin import io, eda, splits, viz_mip

SEED = 42
np.random.seed(SEED)
pd.set_option("display.width", 160)

## 1. Carga del volumen y espaciado físico

Los archivos vienen en **MetaImage `.mha`**, convertidos desde NIfTI, y el header
se quedó con los campos NIfTI originales (`pixdim[1..3]`, `scl_slope`,
`qform/sform`). SimpleITK lee de ahí el espaciado en mm por vóxel, y ese
espaciado es el que uso para **todas** las medidas en milímetros. No asumo
vóxeles isotrópicos en ningún momento: cada caso tiene el suyo.

Tomé dos decisiones en la carga:

**Reorientar todo a LPS** (`sitk.DICOMOrient`). Esta la descubrí comparando
casos: el 001 viene en RAI y el 002 en LPI. No comparten dirección. Si no
reorientara, la mitad de los volúmenes queda espejada y el modelo aprendería la
lateralidad al revés — llamaría coxal izquierdo al derecho. Es el tipo de error
que no se ve en la métrica de entrenamiento y arruina todo lo demás.

**Dejar el array en orden (z, y, x)** y devolver el espaciado en ese mismo
orden. Suena obvio, pero mezclar los dos órdenes es la forma más fácil de
calcular volúmenes mal sin que nada reviente.

In [4]:
CASO_DEMO = sorted(RUTA_CT.keys() & RUTA_LB.keys())[0]
caso = io.cargar_caso(CASO_DEMO, RUTA_CT[CASO_DEMO].parent, DIR_ETIQUETAS)
print(f"Caso {caso.id}: forma (z,y,x) = {caso.ct.shape}")
print(f"Espaciado (z,y,x) = {tuple(round(s, 4) for s in caso.spacing)} mm")
print(f"Dirección original = {caso.direccion_original[::4]} (diagonal) -> reorientado a {io.ORIENTACION}")
print(f"HU: min {caso.ct.min()}, max {caso.ct.max()}")
print("Etiquetas presentes:", {int(v): io.REGIONES[io.region_de_etiqueta(v)]['sigla'] + f"-{io.fragmento_de_etiqueta(v)}"
                               for v in np.unique(caso.etiqueta) if v})

Caso 001: forma (z,y,x) = (401, 512, 512)
Espaciado (z,y,x) = (0.8, 0.7812, 0.7812) mm
Dirección original = (1.0, 1.0, 1.0) (diagonal) -> reorientado a LPS
HU: min -1023, max 2775


Etiquetas presentes: {1: 'SA-1', 11: 'LI-1', 12: 'LI-2', 21: 'RI-1'}


## 2. Curación: control de calidad CT ↔ máscara

Antes de tocar un solo vóxel reviso los headers de los 100 casos. Es barato y
descarta problemas que después cuestan horas.

Para cada caso verifico que el CT y su máscara coincidan en tamaño, espaciado,
origen y dirección; que `scl_slope` sea 1, porque si no los valores no son HU
directos y todo el ventaneo quedaría corrido; y que el `ElementSpacing` del
`.mha` coincida con el `pixdim` del NIfTI original, para confirmar que la
conversión no perdió la escala física.

Los 100 casos pasaron. Lo dejo igual en el pipeline porque el día que agregue
datos nuevos quiero que el chequeo esté ahí y no confiar en que "ya lo revisé
una vez".

In [5]:
def qc_caso(cid):
    fila = {"caso": cid}
    lb = io.leer_header(RUTA_LB[cid])
    fila.update({k: lb[k] for k in ("nx", "ny", "nz", "sx_mm", "sy_mm", "sz_mm")})
    fila["dir_diag"] = str(lb["direccion"])
    if cid in RUTA_CT:
        ct = io.leer_header(RUTA_CT[cid])
        fila["tam_ok"] = (ct["nx"], ct["ny"], ct["nz"]) == (lb["nx"], lb["ny"], lb["nz"])
        fila["spacing_ok"] = np.allclose([ct["sx_mm"], ct["sy_mm"], ct["sz_mm"]],
                                         [lb["sx_mm"], lb["sy_mm"], lb["sz_mm"]], atol=1e-3)
        fila["dir_ok"] = ct["direccion"] == lb["direccion"]
        fila["slope_1"] = ct["scl_slope"] == 1 and ct["scl_inter"] == 0
        r = sitk.ImageFileReader(); r.SetFileName(str(RUTA_CT[cid])); r.ReadImageInformation()
        pix = [float(r.GetMetaData(f"pixdim[{i}]")) for i in (1, 2, 3)] if r.HasMetaDataKey("pixdim[1]") else None
        fila["pixdim_ok"] = pix is None or np.allclose(pix, r.GetSpacing(), atol=1e-4)
    return fila

qc = pd.DataFrame([qc_caso(c) for c in sorted(RUTA_LB)])
cols_ok = [c for c in ("tam_ok", "spacing_ok", "dir_ok", "slope_1", "pixdim_ok") if c in qc]
qc["apto"] = qc[cols_ok].all(axis=1) if cols_ok else True
qc.to_csv(SALIDAS / "qc_headers.csv", index=False)
print(f"Casos revisados: {len(qc)} · con CT: {qc[cols_ok[0]].notna().sum() if cols_ok else 0} · aptos: {int(qc['apto'].sum())}")
print("\nDirecciones (diagonal) encontradas:\n", qc["dir_diag"].value_counts().to_string())
print("\nEspaciado (mm):\n", qc[["sx_mm", "sy_mm", "sz_mm"]].describe().round(3).to_string())
qc[~qc["apto"]]

Casos revisados: 100 · con CT: 3 · aptos: 100

Direcciones (diagonal) encontradas:
 dir_diag
(1, 1, 1)      66
(-1, -1, 1)    34

Espaciado (mm):
          sx_mm    sy_mm    sz_mm
count  100.000  100.000  100.000
mean     0.819    0.819    0.891
std      0.079    0.079    0.109
min      0.658    0.658    0.625
25%      0.781    0.781    0.800
50%      0.798    0.798    0.801
75%      0.858    0.858    1.000
max      1.220    1.220    1.250


,caso,nx,ny,nz,sx_mm,sy_mm,sz_mm,dir_diag,tam_ok,spacing_ok,dir_ok,slope_1,pixdim_ok,apto


## 3. EDA de fragmentos por caso

La taxonomía del dataset (sección 2.1 del enunciado) es: **1–10 sacro (SA)**,
**11–20 coxal izquierdo (LI)**, **21–30 coxal derecho (RI)**. El que termina en
1 es el fragmento principal, y un hueso sin fracturar es un solo fragmento.

Por cada fragmento calculo tres cosas: volumen en mL usando el espaciado del
header, cuántas componentes conexas tiene, y la **distancia mínima al fragmento
principal en milímetros** (`scipy.ndimage.distance_transform_edt` con
`sampling=spacing`). Esa última la saco sobre el ground truth a propósito: es la
referencia contra la que voy a comparar la distancia que prediga el modelo en la
semana 10. Si no la mido ahora, después no tengo con qué comparar.

Corre sobre las 100 máscaras y toma unos 10 minutos en Colab. Si ya existe
`salidas/eda_casos.csv` lo reutiliza, para no repetir el cálculo cada vez que
reabro el notebook.

**Lo que salió:** 100 casos, 575 fragmentos. Entre 3 y 9 fragmentos por caso,
con un promedio de 5,8. De los 100 casos, 39 tienen un hueso fracturado, 44
tienen dos y 16 tienen los tres. Solo un caso no tiene ninguno.

In [6]:
if (SALIDAS / "eda_casos.csv").exists() and (SALIDAS / "eda_fragmentos.csv").exists():
    casos = pd.read_csv(SALIDAS / "eda_casos.csv", dtype={"caso": str})
    frags = pd.read_csv(SALIDAS / "eda_fragmentos.csv", dtype={"caso": str})
else:
    res = [eda.resumen_caso(RUTA_LB[c]) for c in sorted(RUTA_LB)]
    casos = pd.DataFrame([c for c, _ in res]); frags = pd.DataFrame([f for _, fs in res for f in fs])
    casos.to_csv(SALIDAS / "eda_casos.csv", index=False); frags.to_csv(SALIDAS / "eda_fragmentos.csv", index=False)
casos["caso"] = casos["caso"].str.zfill(3); frags["caso"] = frags["caso"].str.zfill(3)
print(f"{len(casos)} casos · {len(frags)} fragmentos · {int((~frags['es_principal']).sum())} fragmentos no principales")
casos.head()

100 casos · 575 fragmentos · 275 fragmentos no principales


,caso,nz,ny,nx,sz_mm,sy_mm,sx_mm,cortes_SA,cortes_LI,cortes_RI,...,vol_ml_SA,principal_es_mayor_SA,n_frag_LI,vol_ml_LI,principal_es_mayor_LI,n_frag_RI,vol_ml_RI,principal_es_mayor_RI,n_frag_total,regiones_fracturadas
0,001,401,512,512,0.8,0.781250,0.781250,170,274,283,...,223.90,True,2,403.39,True,1,412.28,True,4,1
1,002,337,276,413,0.8,0.824219,0.824219,167,246,245,...,197.29,True,1,273.12,True,3,274.39,True,6,2
2,003,285,381,512,1.0,0.759766,0.759766,134,206,210,...,181.40,True,4,295.23,True,1,281.04,True,6,1
3,004,224,254,386,1.0,0.843750,0.843750,137,174,182,...,149.59,True,2,225.11,True,2,224.19,True,6,3
4,005,312,512,512,1.0,0.896484,0.896484,123,192,193,...,156.30,True,2,241.86,True,3,244.60,True,6,2


In [7]:
SIG = [i["sigla"] for i in io.REGIONES.values()]
NOMBRE = {i["sigla"]: i["nombre"] for i in io.REGIONES.values()}
COLOR = {i["sigla"]: i["color"] for i in io.REGIONES.values()}

tabla = pd.DataFrame({
    "casos con la región": [(casos[f"n_frag_{s}"] > 0).sum() for s in SIG],
    "casos fracturados": [(casos[f"n_frag_{s}"] > 1).sum() for s in SIG],
    "fragmentos (media)": [casos[f"n_frag_{s}"].mean().round(2) for s in SIG],
    "fragmentos (máx)": [casos[f"n_frag_{s}"].max() for s in SIG],
    "volumen mL (mediana)": [casos[f"vol_ml_{s}"].median().round(1) for s in SIG],
    "principal = el mayor": [f"{casos[f'principal_es_mayor_{s}'].mean():.0%}" for s in SIG],
}, index=[NOMBRE[s] for s in SIG])
tabla

,casos con la región,casos fracturados,fragmentos (media),fragmentos (máx),volumen mL (mediana),principal = el mayor
Sacro,100,45,1.53,4,187.1,100%
Coxal izquierdo,100,66,2.15,6,287.5,100%
Coxal derecho,100,64,2.07,4,291.6,100%


In [8]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
maxf = int(casos[[f"n_frag_{s}" for s in SIG]].max().max())
bins = np.arange(0.5, maxf + 1.5)
for s in SIG:
    ax[0].hist(casos[f"n_frag_{s}"], bins=bins, alpha=.6, label=NOMBRE[s], color=COLOR[s])
ax[0].set(title="Fragmentos por región y caso", xlabel="n.º de fragmentos", ylabel="casos"); ax[0].legend()

ax[1].hist(casos["regiones_fracturadas"], bins=np.arange(-.5, 4), rwidth=.8, color="gray")
ax[1].set(title="Huesos fracturados por caso", xlabel="regiones con >1 fragmento", ylabel="casos", xticks=range(4))

nop = frags[~frags["es_principal"]]
for s in SIG:
    v = nop.loc[nop["region"] == s, "vol_ml"]
    if len(v): ax[2].hist(np.log10(v), bins=25, alpha=.6, color=COLOR[s], label=NOMBRE[s])
ax[2].set(title="Volumen de fragmentos no principales", xlabel="log10(mL)", ylabel="fragmentos"); ax[2].legend()
fig.tight_layout(); fig.savefig(SALIDAS / "eda_fragmentos.png", dpi=120)

### 3.1 Separación al fragmento principal (ground truth)

`dist_mm_gt` es la distancia mínima entre los vóxeles del fragmento y los del
principal. Cuando dos fragmentos se tocan, el valor da del orden de un vóxel
(0,7–1 mm), así que `en_contacto` marca los que están a una diagonal de vóxel o
menos.

Una aclaración sobre la tabla: de los 575 fragmentos, 300 salen sin valor en
esta columna. No es un dato faltante — son exactamente los 300 fragmentos
principales, y la distancia de un fragmento a sí mismo no significa nada. Los
275 restantes sí tienen su distancia calculada.

In [9]:
nop = frags[~frags["es_principal"]].copy()
nop["en_contacto"] = nop["en_contacto"].astype(str).eq("True")   # el CSV lo lee como texto
print(f"Fragmentos no principales: {len(nop)}")
print(f"En contacto con el principal: {nop['en_contacto'].mean():.1%}")
print(f"Con más de una componente conexa: {(nop['n_componentes'] > 1).mean():.1%}")
print("\nDistancia al principal (mm) por región:")
print(nop.groupby("region")["dist_mm_gt"].describe().round(2).rename(index=NOMBRE).to_string())

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sep = nop[~nop["en_contacto"]]
for s in SIG:
    d = sep.loc[sep["region"] == s, "dist_mm_gt"]
    if len(d): ax[0].hist(d, bins=30, alpha=.6, color=COLOR[s], label=f"{NOMBRE[s]} (n={len(d)})")
ax[0].set(title="Separación de fragmentos que NO tocan al principal", xlabel="mm", ylabel="fragmentos"); ax[0].legend()
ax[1].scatter(nop["vol_ml"], nop["dist_mm_gt"], c=nop["region"].map(COLOR), s=14, alpha=.7)
ax[1].set(xscale="log", title="Volumen vs. separación", xlabel="volumen (mL, log)", ylabel="distancia (mm)")
fig.tight_layout(); fig.savefig(SALIDAS / "eda_separacion.png", dpi=120)

Fragmentos no principales: 275
En contacto con el principal: 88.7%
Con más de una componente conexa: 33.8%

Distancia al principal (mm) por región:
                 count  mean   std   min   25%   50%   75%    max
region                                                           
Coxal izquierdo  115.0  1.88  5.30  0.66  0.78  0.78  0.84  50.82
Coxal derecho    107.0  2.88  8.93  0.62  0.78  0.80  0.83  73.12
Sacro             53.0  1.13  1.72  0.66  0.78  0.80  0.81  11.52


### 3.2 Cortes axiales por región (desbalance 2D)

Acá hay una trampa que vale la pena mirar. El modelo no ve casos, ve **cortes**,
uno por uno. Así que para la cabeza de clasificación lo que importa no es
cuántos casos tienen sacro, sino en cuántos cortes aparece cada región.

Y el desbalance es real: contando cortes, los coxales aparecen bastante más que
el sacro. Por eso lo mido ahora, para tenerlo en cuenta cuando arme el muestreo
de las semanas siguientes en vez de descubrirlo cuando el modelo ignore el
sacro.

In [10]:
cortes = pd.DataFrame({NOMBRE[s]: casos[f"cortes_{s}"] for s in SIG})
cortes["Con algún hueso"] = casos["cortes_con_hueso"]; cortes["Total del volumen"] = casos["nz"]
print(cortes.sum().to_string())
print(f"\nFracción de cortes sin hueso anotado: {1 - casos['cortes_con_hueso'].sum() / casos['nz'].sum():.1%}")
cortes.describe().round(1)

Sacro                16052
Coxal izquierdo      23411
Coxal derecho        23275
Con algún hueso      24316
Total del volumen    32106

Fracción de cortes sin hueso anotado: 24.3%


,Sacro,Coxal izquierdo,Coxal derecho,Con algún hueso,Total del volumen
count,100.0,100.0,100.0,100.0,100.0
mean,160.5,234.1,232.8,243.2,321.1
std,22.7,30.2,31.3,31.4,43.5
min,122.0,164.0,154.0,165.0,193.0
25%,141.0,213.0,206.8,217.8,285.8
50%,162.5,243.5,241.0,251.5,326.0
75%,175.2,255.5,255.2,266.0,350.0
max,217.0,340.0,345.0,347.0,414.0


### 3.3 Intensidad del hueso anotado (justifica ventana y umbral)

Esta sección salió de una duda: ¿sirve el umbral clásico de 200 HU para separar
hueso?

Medí los percentiles de HU dentro de la máscara de referencia y la respuesta es
que no, al menos no como segmentación. En el caso 001, **el 54 % del hueso
anotado está por debajo de 200 HU**. En el 002 es el 41 %, y en el 003 el 15 %.
O sea que si umbralizara a 200 HU, en algunos casos perdería la mitad del hueso
que el experto marcó — sobre todo el trabecular del sacro, que es menos denso
que la cortical.

La conclusión práctica: uso 200 HU **solo para el MIP**, donde funciona porque
la proyección de máxima intensidad se queda con la cortical y eso es justo lo
que quiero ver. Pero a la red le paso una **ventana ancha**, no una máscara
umbralizada, para que no le quite el hueso que sí importa.

Otro detalle que apareció acá: el caso 003 llega a 16.709 HU de máximo. Eso no
es hueso, es metal — material quirúrgico o un artefacto. Lo anoto porque va a
molestar cuando llegue a segmentación.

In [11]:
N_CASOS_HU = 12
disponibles = sorted(RUTA_CT.keys() & RUTA_LB.keys())
muestra = list(np.random.default_rng(SEED).choice(disponibles, min(N_CASOS_HU, len(disponibles)), replace=False))
hu = pd.DataFrame([eda.estadisticas_hu(RUTA_CT[c], RUTA_LB[c]) for c in sorted(muestra)])
hu.to_csv(SALIDAS / "eda_hu_hueso.csv", index=False)
print(f"Hueso anotado bajo 200 HU (media de {len(hu)} casos): {hu['frac_hueso_bajo_200HU'].mean():.1%}")
hu.round(1)

Hueso anotado bajo 200 HU (media de 3 casos): 36.8%


,caso,hu_min,hu_max,hueso_p1,hueso_p5,hueso_p25,hueso_p50,hueso_p75,hueso_p95,hueso_p99,frac_hueso_bajo_200HU
0,001,-1023,2775,-205.0,-97.0,49.0,173.0,421.0,1127.0,1442.0,0.5
1,002,-1023,4711,-17.0,28.0,121.0,252.0,464.0,883.0,1108.0,0.4
2,003,-1023,16709,32.0,110.0,261.0,409.0,649.0,1050.0,1209.0,0.1


## 4. Splits fijos por paciente

La unidad del split es el **caso completo**, y esto no es un detalle menor.
Cortes vecinos del mismo CT son casi idénticos; si reparto cortes entre train y
test, el modelo ve en test algo prácticamente igual a lo que memorizó en train y
las métricas salen infladas. Sería engañarme a mí mismo.

Así que reparto por paciente: 70/15/15, semilla 42, estratificado por cuántos
huesos están fracturados, para que val y test no se queden sin casos
multifragmento. Solo entran los casos que pasaron el QC.

`splits/splits.json` queda congelado en el repositorio. Nadie lo vuelve a
sortear — si cambia el split, los resultados de las semanas anteriores dejan de
ser comparables.

In [12]:
aptos = set(qc.loc[qc["apto"], "caso"]) if "tam_ok" in qc and qc["tam_ok"].notna().all() else set(casos["caso"])
base = casos[casos["caso"].isin(aptos)].copy()
base["estrato"] = splits.estrato_por_complejidad(base)
ruta_splits = RAIZ / "splits/splits.json"
if ruta_splits.exists():
    sp = splits.cargar(ruta_splits); print("splits.json existente (congelado) -> se reutiliza")
else:
    sp = splits.estratificar(base, "estrato")
    splits.guardar(sp, ruta_splits, {"proporciones": [0.70, 0.15, 0.15], "unidad": "caso",
                                     "estrato": "regiones_fracturadas (0-3, fusionando estratos < 7 casos)"})
assert not (set(sp["train"]) & set(sp["val"]) or set(sp["train"]) & set(sp["test"]) or set(sp["val"]) & set(sp["test"]))

resumen = []
for nombre in ("train", "val", "test"):
    d = base[base["caso"].isin(sp[nombre])]
    resumen.append({"split": nombre, "casos": len(d), "cortes": int(d["nz"].sum()),
                    "fragmentos": int(d["n_frag_total"].sum()),
                    **{f"fract. {s}": int((d[f"n_frag_{s}"] > 1).sum()) for s in SIG},
                    "frag/caso": round(d["n_frag_total"].mean(), 2)})
pd.DataFrame(resumen).set_index("split")

,casos,cortes,fragmentos,fract. SA,fract. LI,fract. RI,frag/caso
split,,,,,,,
train,70,22744,396,33,46,44,5.66
val,15,4976,88,7,7,12,5.87
test,15,4386,91,5,13,8,6.07


## 5. Ventaneo HU

`io.ventana_hu(ct, nivel, ancho)` recorta a `[L − W/2, L + W/2]` y normaliza a
[0, 1]. Esa es la entrada de la red.

Uso ventana de hueso con L = 400 y W = 1800, o sea de −500 a 1300 HU. Es ancha a
propósito, por lo que vi en la sección 3.3: si la cierro más, me como el hueso
trabecular del sacro que el experto sí anotó.

In [13]:
# corte con más vóxeles de fragmentos no principales (si no hay fractura, el de más hueso)
no_principal = (caso.etiqueta > 0) & ((caso.etiqueta.astype(int) - 1) % 10 > 0)
z = int(np.argmax(no_principal.sum(axis=(1, 2)) if no_principal.any() else (caso.etiqueta > 0).sum(axis=(1, 2))))
reg = np.vectorize(io.region_de_etiqueta)(caso.etiqueta[z])
from matplotlib.colors import ListedColormap
cmap_reg = ListedColormap(["none"] + [i["color"] for i in io.REGIONES.values()])

fig, ax = plt.subplots(1, 4, figsize=(18, 4.8))
ax[0].imshow(caso.ct[z], cmap="gray"); ax[0].set_title("HU crudo")
ax[1].imshow(io.ventana_hu(caso.ct[z], *io.VENTANAS["tejido_blando"]), cmap="gray"); ax[1].set_title("Ventana tejido blando (40/400)")
ax[2].imshow(io.ventana_hu(caso.ct[z]), cmap="gray"); ax[2].set_title("Ventana hueso (400/1800)")
ax[3].imshow(io.ventana_hu(caso.ct[z]), cmap="gray"); ax[3].imshow(reg, cmap=cmap_reg, vmin=0, vmax=3, alpha=.45, interpolation="nearest")
ax[3].set_title("Ventana hueso + GT por región")
for a in ax: a.axis("off")
fig.suptitle(f"Caso {caso.id} · corte axial z={z} · uso académico, no clínico"); fig.tight_layout()
fig.savefig(SALIDAS / f"ventaneo_{caso.id}.png", dpi=110)

## 6. Visualizador 1 — MIP del volumen crudo (solo hueso)

Este visualizador es preprocesamiento clásico, **sin modelo** — la idea es tener
una referencia de lo que se ve con puro procesamiento de imagen, antes de que
haya red que comparar.

El proceso: umbral de hueso en HU sobre el volumen sin procesar, remuestreo a
vóxel isotrópico de 2 mm (solo para que la proyección no salga deformada, porque
el espaciado original no es isotrópico) y proyección de máxima intensidad.
Girando el volumen alrededor del eje cráneo-caudal se arma la vista 3D: el
slider recorre los ángulos y el botón ▶ lo anima.

Lo exporto como HTML autocontenido para que el dashboard de la semana 11 lo
incruste tal cual, sin dependencias.

In [14]:
UMBRAL_HU = 200
viz_mip.figura_tres_vistas(caso.ct, caso.spacing, caso.id, UMBRAL_HU).savefig(SALIDAS / f"mip3_{caso.id}.png", dpi=110)
fig = viz_mip.figura_mip(caso.ct, caso.spacing, caso.id, umbral_hu=UMBRAL_HU)
fig.write_html(SALIDAS / f"visualizador1_mip_{caso.id}.html", include_plotlyjs="cdn")
fig.show()

## Resumen de la semana

Lo que queda generado en `salidas/` y `splits/`:
`qc_headers.csv`, `eda_casos.csv`, `eda_fragmentos.csv`, `eda_hu_hueso.csv`,
`eda_*.png`, `ventaneo_*.png`, `mip3_*.png`, `visualizador1_mip_*.html`,
`splits/splits.json`.

**Dónde quedé:** 100 casos curados, 575 fragmentos medidos, splits congelados y
un visualizador funcionando sin modelo. Los datos ya están en un estado en el que
puedo construir encima sin sospechar de ellos.

**Lo que me llevo anotado para adelante:** el metal del caso 003 va a dar
problemas en segmentación, y el desbalance de cortes entre sacro y coxales hay
que compensarlo en el muestreo, no ignorarlo.

**Semana 9:** backbone (extensión de `FundidoraPC`) + CBAM + cabeza de detección
con grid y NMS propios, y un overfit intencional sobre un batch pequeño para
confirmar que la arquitectura puede aprender antes de entrenar en serio.